# Exploração e preparação SQL

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Leitura versionada e filtros. Esperado: snapshot, tipos, nulos e amostra; nenhuma escrita.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.data.journey import extract,prepare,QUERY
frame,snapshot=extract(cfg)
display(snapshot)
display(Markdown("```sql\n"+QUERY.read_text()+"\n```"))
display(frame.dtypes.astype(str))
display(frame.head())
display(frame.isna().sum())


{'id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
 'cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
 'target_index': 4052,
 'target_semester': 1,
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'query_sha256': 'defc10d1b6ea752600af271285c54107663e91db9df13914aea7e42613c83adf',
 'query_version': 'journey-history-v1'}

```sql
-- journey-history-v1. Target: total students per period, replicated to its disciplines
-- under the provisional POC rule: every discipline receives the whole cohort.
SELECT h.course_id::text, h.period_id::text, pd.discipline_id::text,
       h.reference_year * 2 + h.reference_semester - 1 AS semester_index,
       h.reference_semester AS semester,
       h.regular_students + h.dependency_students + h.elective_students AS students
FROM planning.historical_enrollments h
JOIN planning.dataset_snapshots s ON s.id = h.snapshot_id
JOIN planning.academic_cycles c ON c.id = s.cycle_id
JOIN planning.period_disciplines pd ON pd.period_id = h.period_id
WHERE h.snapshot_id = CAST(:snapshot AS uuid) AND c.cycle_code = :cycle
  AND s.sealed_at IS NOT NULL
  AND (CAST(:course AS text) IS NULL OR h.course_id::text = :course)
  AND h.reference_year * 2 + h.reference_semester - 1 < c.target_year * 2 + c.target_semester - 1
ORDER BY semester_index, h.course_id, h.period_id, pd.discipline_id

```

course_id         object
period_id         object
discipline_id     object
semester_index     int64
semester           int64
students           int64
dtype: object

,course_id,period_id,discipline_id,semester_index,semester,students
0,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,4040,1,86
1,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,4040,1,86
2,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,4040,1,86
3,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,4040,1,82
4,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,4040,1,82


course_id         0
period_id         0
discipline_id     0
semester_index    0
semester          0
students          0
dtype: int64

Separação temporal sem ajuste de transformações. Esperado: três conjuntos ordenados no tempo.

In [2]:
train,evaluation,future=prepare(frame,snapshot)
display({"train":len(train),"evaluation":len(evaluation),"future":len(future)})


{'train': 960, 'evaluation': 90, 'future': 90}